# Lezione 3 — Da requisiti a raccolta: procurati i dati corretti

**Durata stimata: 120 minuti**

## Obiettivi di apprendimento

- Identificare quale contenuto, formato e fonte di dati serve a un progetto.
- Distinguere dati strutturati, non strutturati e semi-strutturati.
- Caricare un dataset reale in Python con pandas ed esplorarne forma e contenuto.
- Decidere quando rimandare la raccolta di un dato e quando è critica.

## 1. L'analogia: i dati come ingredienti

Se il problema di business è la **ricetta**, i dati sono l'**ingrediente**.

Prima di cucinare devi sapere:

- Quali ingredienti servono (**contenuto**)
- In quale formato ti servono (**formato**)
- Da dove li prenderai (**fonte**)

> Come in cucina, un ingrediente può essere "fuori stagione": più difficile da trovare o più costoso del previsto. Questo è esattamente ciò che si rivede nella fase di Data Collection.

Hai già scelto la ricetta e il tipo di cucina nella [lezione 2](../manual/03_problema_a_enfoque_IT.md) (comprensione del business + albero decisionale come approccio analitico). Ora tocca andare a fare la spesa.

## 2. Data Requirements — caso reale: pazienti con CHF

Obiettivo: definire il **contenuto**, il **formato** e la **fonte** dei dati per l'albero decisionale.

### La coorte di pazienti: 3 criteri di inclusione

1. Ammesso come degente interno nell'area di servizio dell'assicurazione.
2. Diagnosi principale di insufficienza cardiaca congestizia (CHF) durante un anno intero.
3. Almeno 6 mesi di iscrizione continua prima dell'ammissione principale per CHF.

**Sono stati esclusi** i pazienti con CHF che avevano altre condizioni mediche significative: quelle condizioni innescano tassi di ricovero ripetuto più alti e distorcerebbero il modello.

### Il formato richiesto: un record per paziente

- L'albero decisionale ha bisogno di **una riga per paziente**, con colonne = variabili del modello.
- La storia clinica reale porta migliaia di record transazionali per paziente (ammissioni, diagnosi, procedure, prescrizioni...).
- Bisogna aggregare (*rollup*) quei record a livello di paziente, creando nuove variabili riassuntive.
- Quel lavoro di aggregazione appartiene tecnicamente alla fase successiva (Data Preparation), ma viene anticipato qui.

## 3. Data Collection — caso reale: rivedere e decidere

Dopo la raccolta iniziale, tocca chiedersi: **ho ciò che mi serve?**

Fonti usate nel caso CHF:

- Informazioni demografiche (età, genere, ubicazione)
- Informazioni cliniche
- Informazioni sulla copertura assicurativa
- Informazioni del fornitore sanitario
- Registri di rimborsi (*claims*)
- Informazioni farmaceutiche e di altre diagnosi

### Il punto chiave: va bene rimandare un dato

- Alcune informazioni sui **farmaci** non erano ancora integrate con il resto delle fonti.
- Decisione: **rimandare** la loro raccolta, non bloccare il progetto.
- Si può riprendere più tardi, anche dopo aver avuto **risultati intermedi** del modello predittivo, se quei risultati suggeriscono che il dato conta.
- In questo caso reale: il team ha costruito un modello ragionevolmente buono **senza** quell'informazione sui farmaci.

> Non tutti i dati sono ugualmente critici. Sapere quali puoi posticipare è importante quanto sapere quali servono subito.

Una volta raccolti i dati, si applica **statistica descrittiva** e **visualizzazione** per valutare contenuto, qualità e individuare buchi.

## 4. Tre modi in cui arrivano i dati

- **Strutturati**: tabelle, righe e colonne ben definite (foglio di calcolo, tabella di database).
- **Non strutturati**: senza formato predefinito — testo libero, immagini, audio, video. Richiedono tecniche speciali.
- **Semi-strutturati**: organizzazione parziale (etichette, marcatori) ma senza la rigidità di una tabella — ad esempio, ricette con nome + lista variabile di ingredienti.

Riconoscere il tipo di dato ti dice quali strumenti ti serviranno.

## 5. Integrazione dei dati: DBA e programmatori

Quando i dati provengono da fonti diverse (demografici, clinici, copertura, rimborsi...), qualcuno deve estrarli e fonderli.

- Quel lavoro lo fanno i **DBA** (amministratori di database) insieme ai programmatori.
- Compito: estrarre, fondere, ed **eliminare la ridondanza**.
- Il risultato resta pronto per la fase successiva: Data Understanding.
- È anche il momento di pensare ad automatizzare processi di database per raccolte future.

## 6. Il caso pratico di questa lezione: 57.691 ricette di cucina

Da qui in poi lavoriamo con dati reali e scaricabili (il caso CHF non ha dati pubblici disponibili).

- Ricercatore: **Yong-Yeol Ahn**.
- Raccolta: *web scraping* (estrazione automatica di dati da pagine web) da tre siti: allrecipes.com, epicurious.com, menupan.com.
- Domanda di business: **si può automatizzare la determinazione della cucina (*cuisine*) di una ricetta a partire dai suoi ingredienti?**

**Buona notizia:** non serve fare scraping. I dati sono già raccolti e disponibili su un server di IBM. Basta scaricarli e iniziare a esplorarli — Data Collection, in questo caso, è già risolta.

## 7. Caricare il dataset con pandas

- **pandas**: la libreria standard di Python per lavorare con dati in tabelle.
- **import**: istruzione che porta il codice di una libreria per poterla usare.
- **DataFrame**: l'oggetto centrale di pandas — una tabella in memoria, con righe e colonne, come un foglio di calcolo ma manipolabile con codice.

In [ ]:
import pandas as pd  # porta pandas e gli dà l'alias "pd", per convenzione

# Chiede a pandas di mostrare tutte le colonne quando stampa la tabella,
# invece di tagliare le colonne centrali con i puntini di sospensione.
pd.set_option('display.max_columns', None)

# pd.read_csv() scarica il file CSV dall'URL e lo converte
# direttamente in un DataFrame: una tabella di righe e colonne in memoria.
recipes = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0103EN-SkillsNetwork/labs/Module%202/recipes.csv")

print("Data read into dataframe!")  # richiede circa 30 secondi per scaricare

In [ ]:
recipes.head()  # mostra le prime 5 righe della tabella, per vederne la forma

In [ ]:
recipes.shape  # restituisce (numero di righe, numero di colonne)

Il dataset ha **57.691 ricette**. Ogni riga è una ricetta, e porta documentata la sua cucina di origine, oltre a se ognuno dei 384 ingredienti distinti compare o no in quella ricetta (da "almond" a "zucchini").

## 8. Un primo sguardo aggiuntivo

Esploriamo un po' di più la forma del dataset, senza ancora assumere nomi esatti di colonne — questo, e la pulizia della colonna della cucina, è tema della prossima lezione (Data Understanding / Data Preparation).

In [ ]:
recipes.columns[:10]  # i nomi delle prime 10 colonne del DataFrame

In [ ]:
# La prima colonna documenta la cucina di ogni ricetta.
# Con .value_counts() contiamo quante ricette ci sono per ogni valore,
# e con .head(10) teniamo i 10 più frequenti.
recipes.iloc[:, 0].value_counts().head(10)

## Riepilogo della lezione

- I dati sono un ingrediente: prima definisci contenuto, formato e fonte (Data Requirements), poi valuti e aggiusti ciò che hai raccolto (Data Collection).
- Caso CHF: coorte con 3 criteri di inclusione, esclusione di comorbilità gravi, e formato di un record per paziente (rollup di migliaia di record transazionali).
- Va bene rimandare un dato non disponibile (i dati sui farmaci) e decidere più tardi se serve, anche dopo aver visto risultati intermedi.
- Dati strutturati, non strutturati e semi-strutturati: tre forme diverse di organizzazione.
- DBA e programmatori integrano dati da diverse fonti, eliminando la ridondanza.
- Caso pratico: 57.691 ricette raccolte da Yong-Yeol Ahn via web scraping, già disponibili su un server di IBM, caricate con `pandas.read_csv()`.

**Prossima lezione:** da comprensione a preparazione — statistica descrittiva, qualità dei dati e pulizia con pandas su questo stesso dataset di ricette.